In [0]:
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

# Config
CATALOG = "oil_gas_demo"
SILVER = f"{CATALOG}.silver.supply_chain_transactions"
GOLD_AGG = f"{CATALOG}.gold.daily_demand_agg"


class DailyDemandAggregator:
    """Aggregates silver data into daily demand by product and city."""

    def __init__(self, spark, source_table: str, target_table: str):
        self.spark = spark
        self.source_table = source_table
        self.target_table = target_table

    def write_aggregated_table(self):
        """Create or replace the gold aggregate table using SQL."""
        logger.info(f"Aggregating {self.source_table} -> {self.target_table}")
        self.spark.sql(f"""
            CREATE OR REPLACE TABLE {self.target_table} AS
            SELECT
                txn_date AS transaction_date,
                product_name,
                destination_city,
                ROUND(SUM(CAST(demand_quantity AS DOUBLE)), 2) AS total_demand,
                ROUND(AVG(CAST(unit_price_usd AS DOUBLE)), 2) AS avg_unit_price,
                SUM(CAST(available_inventory AS DOUBLE)) AS total_inventory,
                COUNT(*) AS transaction_count
            FROM {self.source_table}
            GROUP BY txn_date, product_name, destination_city
        """)
        count = self.spark.table(self.target_table).count()
        logger.info(f"Gold table written: {self.target_table} ({count:,} rows)")

    def run(self):
        """Aggregate silver -> gold."""
        self.write_aggregated_table()


# Execute
aggregator = DailyDemandAggregator(
    spark=spark,
    source_table=SILVER,
    target_table=GOLD_AGG,
)
aggregator.run()